In [34]:
!pip install -q -U autogen-agentchat autogen-ext[openai] nest_asyncio

Before debugging

In [ ]:
# Incorrect Code
from autogen import ConversableAgent

agent1 = ConversableAgent(
    name="Agent1",
    llm_config={"model": "gemini-pro"},
)

agent2 = ConversableAgent(
    name="Agent2",
    llm_config={"model": "gemini-pro"},
)

response = agent1.initiate_chat(agent2, message="Hi, calculate 5 * 3", max_turns=2)
print(response)


After debugging

In [35]:
import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

In [36]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",

    include_name_in_message=False,

    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
        "family": "unknown",
    },
)

In [37]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination

termination = MaxMessageTermination(max_messages=4)

team = RoundRobinGroupChat(
    participants=[agent1, agent2],
    termination_condition=termination,
)

# Run the conversation
result = await team.run(task="Hi, calculate 5 * 3")

# Print only the conversation
print("=" * 40)
for msg in result.messages:
    print(f"{msg.source}: {msg.content}")
print("=" * 40)

user: Hi, calculate 5 * 3
Agent1: I'm Agent1, passing the request to Agent2.

Agent2: The result of 5 * 3 is 15.
Agent2: 15
Agent1: Agent1 confirming: The calculation is verified. Agent2's result of 5 * 3 equals 15 is correct.


                    USER
                      │
                      ▼
        "Hi, calculate 5 * 3"
                      │
                      ▼
        ┌─────────────────────────┐
        │        Agent1           │
        │ Coordinator / Planner   │
        └─────────────────────────┘
                      │
                      │ Receives the request
                      │
                      ▼
        "Passing the request..."
                      │
                      ▼
        ┌─────────────────────────┐
        │        Agent2           │
        │ Math Expert / Worker    │
        └─────────────────────────┘
                      │
                      │ Calculates
                      ▼
                5 × 3 = 15
                      │
                      ▼
        Returns answer to Agent1
                      │
                      ▼
        ┌─────────────────────────┐
        │        Agent1           │
        │ Verification Agent      │
        └─────────────────────────┘
                      │
                      │ Verifies result
                      ▼
          "Calculation verified"
                      │
                      ▼
                  FINAL OUTPUT

Sequential 3-Agent Workflow (Groq + AutoGen 0.7.5)

In [ ]:
# Incorrect Code
from autogen import ConversableAgent

agent1 = ConversableAgent(name="Agent1")
agent2 = ConversableAgent(name="Agent2")
agent3 = ConversableAgent(name="Agent3")

response1 = agent1.generate_reply(messages=[{"role": "user", "content": "What is AI?"}])
response2 = agent2.generate_reply(messages=[{"role": "user", "content": response1}])
response3 = agent3.generate_reply(messages=[{"role": "user", "content": response2}])

print(response3)


After debugging

In [38]:
from autogen_agentchat.agents import AssistantAgent

# Agent 1
agent1 = AssistantAgent(
    name="Agent1",
    model_client=model_client,
    system_message="You are an AI expert. Explain the topic clearly."
)

# Agent 2
agent2 = AssistantAgent(
    name="Agent2",
    model_client=model_client,
    system_message="Improve and simplify the previous response."
)

# Agent 3
agent3 = AssistantAgent(
    name="Agent3",
    model_client=model_client,
    system_message="Summarize the final answer in 5 lines."
)

# Agent 1
result1 = await agent1.run(task="What is AI?")
response1 = result1.messages[-1].content

print("Agent1:")
print(response1)

print("-" * 50)

# Agent 2
result2 = await agent2.run(task=response1)
response2 = result2.messages[-1].content

print("Agent2:")
print(response2)

print("-" * 50)

# Agent 3
result3 = await agent3.run(task=response2)
response3 = result3.messages[-1].content

print("Agent3:")
print(response3)

Agent1:
**Introduction to Artificial Intelligence (AI)**

Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as:

* Learning
* Problem-solving
* Reasoning
* Perception
* Understanding language

**Key Characteristics of AI**
---------------------------

1. **Intelligence**: AI systems can analyze data, learn from experience, and make decisions based on that data.
2. **Autonomy**: AI systems can operate independently, making decisions without human intervention.
3. **Adaptability**: AI systems can adapt to new situations and learn from experience.

**Types of AI**
----------------

1. **Narrow or Weak AI**: Designed to perform a specific task, such as image recognition, language translation, or playing chess.
2. **General or Strong AI**: A hypothetical AI system that possesses human-like intelligence and can perform any intellectual task.
3. **Superintelligence**: A hypothetical AI system

                           USER
                             │
                             │
          "What is Artificial Intelligence?"
                             │
                             ▼
        ┌──────────────────────────────────┐
        │          Agent 1                 │
        │    Content Creator / Expert      │
        └──────────────────────────────────┘
                     │
                     │ Creates a detailed explanation
                     ▼
        ┌──────────────────────────────────┐
        │          Agent 2                 │
        │     Editor / Content Improver    │
        └──────────────────────────────────┘
                     │
                     │ Improves grammar,
                     │ readability and structure
                     ▼
        ┌──────────────────────────────────┐
        │          Agent 3                 │
        │       Summary Specialist         │
        └──────────────────────────────────┘
                     │
                     │ Creates concise summary
                     ▼
               FINAL OUTPUT

        User
          │
          ▼
        Agent1 (Explain)
          │
          ▼
        Agent2 (Improve)
          │
          ▼
        Agent3 (Summarize)

Three-Agent Sequential Workflow

In [ ]:
# Incorrect code
from autogen import ConversableAgent

config = {
    "model": "gemini-1.5-pro-latest",
    "api_key": "your_api_key",
    "api_type": "google"
}

thinker = ConversableAgent(name="Thinker", llm_config=config)
analyst = ConversableAgent(name="Analyst", llm_config=config)
decision_maker = ConversableAgent(name="DecisionMaker", llm_config=config)

res1 = thinker.generate_reply(messages=[{"role": "user", "content": "Suggest an innovative startup idea in health tech"}])
idea = res1["content"]

res2 = analyst.generate_reply(messages=[{"role": "user", "content": f"Analyze pros and cons:\n{idea}"}])
analysis = res2["content"]

res3 = decision_maker.generate_reply(messages=[{"role": "user", "content": f"Should we pursue this?\n{analysis}"}])
decision = res3["content"]

print(idea, analysis, decision)


After debugging

In [39]:
from autogen_agentchat.agents import AssistantAgent

# -----------------------------
# Thinker Agent
# -----------------------------
thinker = AssistantAgent(
    name="Thinker",
    model_client=model_client,
    system_message="""
You are a creative entrepreneur.
Generate innovative startup ideas.
"""
)

# -----------------------------
# Analyst Agent
# -----------------------------
analyst = AssistantAgent(
    name="Analyst",
    model_client=model_client,
    system_message="""
You are a business analyst.

Analyze the startup idea.

Include:
- Pros
- Cons
- Market Potential
- Risks
"""
)

# -----------------------------
# Decision Maker Agent
# -----------------------------
decision_maker = AssistantAgent(
    name="DecisionMaker",
    model_client=model_client,
    system_message="""
You are an experienced startup investor.

Based on the analysis,
decide whether the startup should be pursued.

Give:
- Decision (Yes/No)
- Reason
- Confidence (%)
"""
)

# -----------------------------
# Step 1 - Thinker
# -----------------------------
result1 = await thinker.run(
    task="Suggest an innovative startup idea in health tech."
)

idea = result1.messages[-1].content

# -----------------------------
# Step 2 - Analyst
# -----------------------------
result2 = await analyst.run(
    task=f"Analyze the following startup idea:\n\n{idea}"
)

analysis = result2.messages[-1].content

# -----------------------------
# Step 3 - Decision Maker
# -----------------------------
result3 = await decision_maker.run(
    task=f"Should we pursue this startup?\n\n{analysis}"
)

decision = result3.messages[-1].content

# -----------------------------
# Print Results
# -----------------------------
print("="*80)
print("💡 THINKER")
print("="*80)
print(idea)

print("\n" + "="*80)
print("📊 ANALYST")
print("="*80)
print(analysis)

print("\n" + "="*80)
print("✅ DECISION MAKER")
print("="*80)
print(decision)

💡 THINKER
Here's an innovative startup idea in health tech:

**Startup Idea:** Personalized Nutrition and Gut Health Platform using AI-powered Microbiome Analysis

**Platform Name:** GutGenie

**Description:** GutGenie is an AI-driven health tech platform that provides personalized nutrition recommendations and gut health insights based on an individual's unique microbiome profile. The platform leverages cutting-edge microbiome analysis, machine learning algorithms, and data analytics to offer tailored dietary advice, helping users optimize their gut health, prevent chronic diseases, and improve overall well-being.

**Key Features:**

1. **At-home Microbiome Testing Kit:** Users receive a non-invasive, at-home testing kit to collect a stool sample, which is then sent to a lab for microbiome analysis.
2. **AI-powered Microbiome Analysis:** The platform uses machine learning algorithms to analyze the user's microbiome data, identifying patterns and correlations with various health outcom

                              USER
                                │
                                │
     "Suggest an innovative startup idea in health tech"
                                │
                                ▼
                 ┌──────────────────────────┐
                 │      THINKER 💡          │
                 │ Startup Idea Generator   │
                 └──────────────────────────┘
                                │
                                │ Generates startup idea
                                ▼
                 GutGenie Startup Proposal
                                │
                                ▼
                 ┌──────────────────────────┐
                 │      ANALYST 📊          │
                 │ Business Analyst         │
                 └──────────────────────────┘
                                │
                Analyzes the startup
                • Pros
                • Cons
                • Market Potential
                • Risks
                                │
                                ▼
                 Business Analysis Report
                                │
                                ▼
                 ┌──────────────────────────┐
                 │  DECISION MAKER ✅       │
                 │ Startup Investor         │
                 └──────────────────────────┘
                                │
                 Makes Final Decision
                 • YES / NO
                 • Reason
                 • Confidence %
                                │
                                ▼
                         FINAL OUTPUT

In [ ]:
                User
                  │
                  ▼
      ┌────────────────────┐
      │  Thinker Agent 💡   │
      └────────────────────┘
                  │
                  ▼
      ┌────────────────────┐
      │ Analyst Agent 📊   │
      └────────────────────┘
                  │
                  ▼
      ┌────────────────────┐
      │Decision Maker ✅   │
      └────────────────────┘
                  │
                  ▼
            Final Decision

AutoGen 0.7.5 + Groq version

In [41]:
# Incorrect Code
from autogen import ConversableAgent

config = {
    "model": "models/gemini-1.5-pro-latest",
    "api_type": "google",
    "api_key": "AIza..."
}

reviewer = ConversableAgent(name="Reviewer", llm_config=config)
editor = ConversableAgent(name="Editor", llm_config=config)
summarizer = ConversableAgent(name="Summarizer", llm_config=config)

abstract = "This research explores the use of transformers in low-resource languages."
review = reviewer.generate_reply(messages=[{"role": "user", "content": abstract}])

edited = editor.generate_reply(messages=[{"role": "user", "content": review}])
summary = summarizer.generate_reply(messages=[{"role": "user", "content": edited}])

print("Edited:", edited["content"])
print("Summary:", summary["content"])


ModuleNotFoundError: No module named 'autogen'

After debugging

In [40]:
from autogen_agentchat.agents import AssistantAgent

# -----------------------------
# Reviewer Agent
# -----------------------------
reviewer = AssistantAgent(
    name="Reviewer",
    model_client=model_client,
    system_message="""
You are a research paper reviewer.

Review the given abstract and provide:
- Strengths
- Weaknesses
- Suggestions for improvement
"""
)

# -----------------------------
# Editor Agent
# -----------------------------
editor = AssistantAgent(
    name="Editor",
    model_client=model_client,
    system_message="""
You are an academic editor.

Rewrite the review into a polished,
professional version with proper grammar
and formatting.
"""
)

# -----------------------------
# Summarizer Agent
# -----------------------------
summarizer = AssistantAgent(
    name="Summarizer",
    model_client=model_client,
    system_message="""
You are a research assistant.

Summarize the edited review
into 5 concise bullet points.
"""
)

# ------------------------------------------------
# Research Abstract
# ------------------------------------------------

abstract = """
This research explores the use of transformers
in low-resource languages.
"""

# ------------------------------------------------
# Step 1 : Reviewer
# ------------------------------------------------

review_result = await reviewer.run(task=abstract)
review = review_result.messages[-1].content

# ------------------------------------------------
# Step 2 : Editor
# ------------------------------------------------

edit_result = await editor.run(task=review)
edited = edit_result.messages[-1].content

# ------------------------------------------------
# Step 3 : Summarizer
# ------------------------------------------------

summary_result = await summarizer.run(task=edited)
summary = summary_result.messages[-1].content

# ------------------------------------------------
# Output
# ------------------------------------------------

print("=" * 80)
print("📝 REVIEW")
print("=" * 80)
print(review)

print("\n" + "=" * 80)
print("✏️ EDITED REVIEW")
print("=" * 80)
print(edited)

print("\n" + "=" * 80)
print("📄 SUMMARY")
print("=" * 80)
print(summary)

📝 REVIEW
**Abstract Review**

The given abstract is brief and to the point, but it lacks details and clarity. Here's a review of the abstract:

**Strengths:**

1. **Relevant topic**: The use of transformers in low-resource languages is a relevant and significant area of research in natural language processing.
2. **Concise language**: The abstract is brief and uses simple language, making it easy to understand.

**Weaknesses:**

1. **Lack of specificity**: The abstract does not provide any specific details about the research, such as the languages being studied, the type of transformers being used, or the research questions being addressed.
2. **No clear research objective**: The abstract does not clearly state the objective of the research or what the study aims to achieve.
3. **No indication of methodology**: The abstract does not provide any information about the research methodology, such as the data being used, the experiments being conducted, or the evaluation metrics being emplo

        Research Paper
                │
                ▼
        ┌──────────────────────┐
        │ Reviewer Agent       │
        │ Review               │
        └──────────────────────┘
                │
                ▼
        ┌──────────────────────┐
        │ Editor Agent         │
        │ Improve              │
        └──────────────────────┘
                │
                ▼
        ┌──────────────────────┐
        │ Summarizer Agent     │
        │ Executive Summary    │
        └──────────────────────┘
                │
                ▼
                Final Report

In [42]:
# Incorrect Code
from autogen import ConversableAgent

config = {
    "config_list": [
        {
            "api_key": "your-gemini-api-key",
            "api_type": "google"
        }
    ]
}

question_agent = ConversableAgent(name="QuestionAgent", llm_config=config)
research_agent = ConversableAgent(name="ResearchAgent", llm_config=config)
explainer_agent = ConversableAgent(name="ExplainerAgent", llm_config=config)

q = question_agent.generate_reply(messages=[{"role": "user", "content": "What is Quantum Computing?"}])
r = research_agent.generate_reply(messages=[{"role": "user", "content": q}])
e = explainer_agent.generate_reply(messages=[{"role": "user", "content": r}])

print("Final simplified answer:", e)


ModuleNotFoundError: No module named 'autogen'

Question → Research → Explain Workflow

After debuggin

In [43]:
from autogen_agentchat.agents import AssistantAgent

# =====================================
# Question Agent
# =====================================

question_agent = AssistantAgent(
    name="QuestionAgent",
    model_client=model_client,
    system_message="""
You are an AI Question Expert.

Understand the user's question and expand it
into a detailed research question.
"""
)

# =====================================
# Research Agent
# =====================================

research_agent = AssistantAgent(
    name="ResearchAgent",
    model_client=model_client,
    system_message="""
You are a Research Scientist.

Provide a detailed technical explanation.

Include:
• Definition
• Working Principle
• Advantages
• Disadvantages
• Applications
"""
)

# =====================================
# Explainer Agent
# =====================================

explainer_agent = AssistantAgent(
    name="ExplainerAgent",
    model_client=model_client,
    system_message="""
You are an AI Teacher.

Convert technical content into simple,
easy-to-understand language suitable for beginners.

Use bullet points.
"""
)

# =====================================
# Step 1 : Question Agent
# =====================================

result1 = await question_agent.run(
    task="What is Quantum Computing?"
)

question = result1.messages[-1].content

# =====================================
# Step 2 : Research Agent
# =====================================

result2 = await research_agent.run(
    task=question
)

research = result2.messages[-1].content

# =====================================
# Step 3 : Explainer Agent
# =====================================

result3 = await explainer_agent.run(
    task=research
)

final_answer = result3.messages[-1].content

# =====================================
# Output
# =====================================

print("=" * 80)
print("❓ QUESTION AGENT")
print("=" * 80)
print(question)

print("\n" + "=" * 80)
print("🔬 RESEARCH AGENT")
print("=" * 80)
print(research)

print("\n" + "=" * 80)
print("👨‍🏫 EXPLAINER AGENT")
print("=" * 80)
print(final_answer)

❓ QUESTION AGENT
**Detailed Research Question:**

What is the fundamental concept of quantum computing, including its underlying principles, mechanisms, and architectures, and how does it differ from classical computing in terms of processing power, algorithmic complexity, and potential applications, particularly in fields such as cryptography, optimization, and artificial intelligence, and what are the current challenges and limitations in developing and implementing quantum computers, as well as the potential impact of quantum computing on various industries and societal aspects, including cybersecurity, healthcare, finance, and education?

**Sub-Questions:**

1. **Principles of Quantum Computing:** What are the basic principles of quantum mechanics, such as superposition, entanglement, and wave function collapse, and how are they applied to quantum computing?
2. **Quantum Computing Architectures:** What are the different types of quantum computing architectures, including gate-based

                 User
                   │
                   ▼
      ┌──────────────────────┐
      │ Question Agent ❓     │
      │ Expand the question   │
      └──────────────────────┘
                   │
                   ▼
      ┌──────────────────────┐
      │ Research Agent 🔬     │
      │ Technical explanation │
      └──────────────────────┘
                   │
                   ▼
      ┌──────────────────────┐
      │ Explainer Agent 👨‍🏫 │
      │ Simplify for beginners│
      └──────────────────────┘
                   │
                   ▼
             Final Answer

In [44]:
# Incorrect Code
from autogen import ConversableAgent

config = {
    "config_list": [
        {
            "model": "models/gemini-1.5-pro-latest",
            "api_key": "YOUR_API_KEY",
            "api_type": "google"
        }
    ]
}

summarizer = ConversableAgent(name="Summarizer", llm_config=config)
translator = ConversableAgent(name="Translator", llm_config=config)

summary = summarizer.generate_reply(
    messages=[{"role": "user", "content": "Summarize the article about global warming"}]
)
print("Summary:", summary["content"])

translation = translator.generate_reply(
    messages=[{"role": "user", "content": summary}]
)
print("Translation:", translation["content"])


ModuleNotFoundError: No module named 'autogen'

After debuggin

In [45]:
from autogen_agentchat.agents import AssistantAgent

# ===========================================
# Summarizer Agent
# ===========================================

summarizer = AssistantAgent(
    name="Summarizer",
    model_client=model_client,
    system_message="""
You are an expert summarizer.

Summarize the given article in
5 concise bullet points.
"""
)

# ===========================================
# Translator Agent
# ===========================================

translator = AssistantAgent(
    name="Translator",
    model_client=model_client,
    system_message="""
You are a professional translator.

Translate the given text into Tamil.
Keep the meaning unchanged.
"""
)

# ===========================================
# Step 1 : Summarize
# ===========================================

result1 = await summarizer.run(
    task="Summarize the article about global warming."
)

summary = result1.messages[-1].content

# ===========================================
# Step 2 : Translate
# ===========================================

result2 = await translator.run(
    task=summary
)

translation = result2.messages[-1].content

# ===========================================
# Output
# ===========================================

print("=" * 80)
print("📝 SUMMARY")
print("=" * 80)
print(summary)

print("\n" + "=" * 80)
print("🌍 TAMIL TRANSLATION")
print("=" * 80)
print(translation)

📝 SUMMARY
Here are 5 concise bullet points summarizing the article about global warming:

* Global warming refers to the long-term rise in Earth's average surface temperature, primarily caused by human activities that release greenhouse gases, such as carbon dioxide and methane.
* The main causes of global warming include burning fossil fuels, deforestation, and land-use changes, which lead to an increase in atmospheric carbon dioxide and other greenhouse gases.
* The effects of global warming include rising sea levels, more frequent and severe heatwaves, droughts, and storms, as well as altered ecosystems and loss of biodiversity.
* If left unchecked, global warming is projected to have devastating consequences, including food and water shortages, displacement of populations, and increased risk of extreme weather events.
* To mitigate global warming, individuals and governments can take action by reducing carbon emissions, investing in renewable energy, increasing energy efficiency, a

              User
                │
                ▼
     ┌─────────────────────┐
     │ Summarizer Agent 📝 │
     │ Summarize Article   │
     └─────────────────────┘
                │
                ▼
     ┌─────────────────────┐
     │ Translator Agent 🌍 │
     │ English → Tamil     │
     └─────────────────────┘
                │
                ▼
          Final Output

In [46]:
# Incorrect Code
from autogen import ConversableAgent

config = {
    "config_list": [
        {
            "model": "models/gemini-1.5-pro-latest",
            "api_key": "AIzaSyXXXX",
            "api_type": "google"
        }
    ]
}

reviewer = ConversableAgent(name="Reviewer", llm_config=config)
translator = ConversableAgent(name="Translator", llm_config=config)
assigner = ConversableAgent(name="TaskAssigner", llm_config=config)

review = reviewer.generate_reply(messages=[{"role": "user", "content": "Review this sentence: The cat are on the table."}])
translated = translator.generate_reply(messages=[{"role": "user", "content": review}])
task = assigner.generate_reply(messages=[{"role": "user", "content": translated}])

print("Reviewer:", review)
print("Translator:", translated)
print("TaskAssigner:", task)


ModuleNotFoundError: No module named 'autogen'

After Debugging

In [47]:
from autogen_agentchat.agents import AssistantAgent

# ==========================================
# Reviewer Agent
# ==========================================

reviewer = AssistantAgent(
    name="Reviewer",
    model_client=model_client,
    system_message="""
You are an English Grammar Expert.

Review the given sentence.

Return:
1. Grammar mistakes
2. Correct sentence
3. Short explanation
"""
)

# ==========================================
# Translator Agent
# ==========================================

translator = AssistantAgent(
    name="Translator",
    model_client=model_client,
    system_message="""
You are a professional Tamil translator.

Translate the entire review into Tamil.

Keep the formatting.
"""
)

# ==========================================
# Task Assigner Agent
# ==========================================

assigner = AssistantAgent(
    name="TaskAssigner",
    model_client=model_client,
    system_message="""
You are a Task Manager.

Based on the translated review,
create a small action plan.

Output:

• Task
• Priority
• Deadline
"""
)

# ==========================================
# Step 1 : Review
# ==========================================

result1 = await reviewer.run(
    task="Review this sentence: The cat are on the table."
)

review = result1.messages[-1].content

# ==========================================
# Step 2 : Translate
# ==========================================

result2 = await translator.run(
    task=review
)

translated = result2.messages[-1].content

# ==========================================
# Step 3 : Task Assignment
# ==========================================

result3 = await assigner.run(
    task=translated
)

task = result3.messages[-1].content

# ==========================================
# Output
# ==========================================

print("="*80)
print("📝 REVIEWER")
print("="*80)
print(review)

print("\n" + "="*80)
print("🌍 TRANSLATOR")
print("="*80)
print(translated)

print("\n" + "="*80)
print("📋 TASK ASSIGNER")
print("="*80)
print(task)

📝 REVIEWER
Here's the review:

1. Grammar mistakes: The subject-verb agreement is incorrect. The verb "are" is a plural verb, but the subject "cat" is a singular noun.

2. Correct sentence: The cat is on the table.

3. Short explanation: In English, a singular subject (like "cat") must be paired with a singular verb (like "is"), while a plural subject (like "cats") would be paired with a plural verb (like "are"). In this case, since "cat" is singular, the correct verb to use is "is".

🌍 TRANSLATOR
திருத்தம்:

1. இலக்கண பிழைகள்: சந்தி-வினை பொருத்தம் தவறானது. "are" என்பது பன்மை வினைச்சொல், ஆனால் "cat" என்பது ஒரு தனிப்பட்ட பெயர்ச்சொல்.

2. சரியான வாக்கியம்: பூனை மேசையில் உள்ளது.

3. குறுகிய விளக்கம்: ஆங்கிலத்தில், ஒரு தனிப்பட்ட பொருள் (பூனை போல) ஒரு தனிப்பட்ட வினை (போல "is") உடன் இணைக்கப்பட வேண்டும், அதே போல் ஒரு பன்மை பொருள் (பூனைகள் போல) ஒரு பன்மை வினை (போல "are") உடன் இணைக்கப்படும். இந்த வழக்கில், "பூனை" ஒரு தனிப்பட்டது என்பதால், பயன்படுத்த சரியான வினை "is" ஆகும்.

📋 TASK ASSIGNER
• Ta

              User
                │
                ▼
     ┌─────────────────────┐
     │ Reviewer Agent 📝   │
     │ Grammar Review      │
     └─────────────────────┘
                │
                ▼
     ┌─────────────────────┐
     │ Translator Agent 🌍 │
     │ English → Tamil     │
     └─────────────────────┘
                │
                ▼
     ┌─────────────────────┐
     │ Task Assigner 📋    │
     │ Create Action Plan  │
     └─────────────────────┘
                │
                ▼
            Final Output